# MPS：GPU 为什么能共享

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rootsongjc/gpu-lab-notebooks/blob/main/labs/zh/lab-09-mps.ipynb)

动手理解多进程共享 GPU。先用两个独立进程观察默认的 time-slicing 行为与每个进程的 context 开销，再尝试启动 CUDA MPS 让 kernel 并发，并讲清 MPS 解决了什么、没解决什么、HAMi 为什么没有直接采用它。

本 notebook 配合教程章节 [MPS：GPU 为什么能共享](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-09-mps/) 使用。运行前请确认 **Runtime → Change runtime type → T4 GPU**。

每个 cell 对应教程中一个 Step；Markdown 单元摘录了必要的解释，完整上下文请回到教程阅读。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

## Why this problem matters

[Lab 08](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-08-cuda-kernel/) 的 kernel 是单进程独占整张 GPU。但真实平台里，推理服务常常一个进程只用 30% 算力，独占整卡太浪费。**GPU 共享**就是“让多个进程/容器在同一张卡上合理共存”。MPS 是这条路上绕不开的基础概念，但它**不是 HAMi 的默认选择**。理解这个差异，你才算真正懂 HAMi 在做什么。

## 三种“共存”机制的底层区别

![三种共存机制：默认 time-slicing 下多个 context 轮流独占 GPU、MPS 下多进程 kernel 并发、HAMi 在 CUDA API 层按配额节流](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-09-mps/gpu-sharing-three-modes.svg)

| 机制 | 并发方式 | 显存 | 故障隔离 | 配额 |
| :--- | :--- | :--- | :--- | :--- |
| 默认 time-slicing（多 context） | 驱动按时间片轮流让一个 context 独占 SM；kernel **不并发** | 每进程独立 context，各自占用，**无上限** | 一个进程崩溃不影响其它 | 无 |
| **MPS** | 多个进程的 kernel 经由 MPS server **同时**在 SM 上执行 | 共享一个服务端 context，省掉每进程的 context 开销 | 弱：Volta+ 有按客户端的地址空间隔离，但 server 故障影响全部 | 有但粗糙：环境变量 `CUDA_MPS_ACTIVE_THREAD_PERCENTAGE`、`CUDA_MPS_PINNED_DEVICE_MEM_LIMIT`，启动时设定，无调度器参与 |
| **HAMi-core** | 不改变驱动调度；在用户态 **拦截 CUDA API**，按配额节流 kernel 发射、拒绝超额 `cudaMalloc` | 每容器 `gpumem` 硬上限 | 进程级，与 time-slicing 相同 | `gpumem` / `gpucores`，由 Kubernetes 调度器分配 |

## Step 1：两个进程，各自的 context

先写一个会持续跑 kernel 的小脚本（复用 [Lab 08](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-08-cuda-kernel/) 的 matmul）：

In [ ]:
%%writefile worker.py
import sys, time, torch
name, seconds = sys.argv[1], float(sys.argv[2])
a = torch.randn(4096, 4096, device="cuda"); b = torch.randn(4096, 4096, device="cuda")
torch.cuda.synchronize()
t0 = time.time(); n = 0
while time.time() - t0 < seconds:
    c = a @ b; n += 1
torch.cuda.synchronize()
print(f"{name}: {n} matmuls in {seconds:.0f}s -> {n/seconds:.1f} it/s", flush=True)

单独跑一个，记下基线吞吐：

In [ ]:
!python worker.py solo 10
!nvidia-smi --query-gpu=memory.used --format=csv

然后同时跑两个：

In [ ]:
import subprocess, time
p1 = subprocess.Popen(["python", "worker.py", "A", "15"])
p2 = subprocess.Popen(["python", "worker.py", "B", "15"])
time.sleep(8)
!nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv
!nvidia-smi --query-compute-apps=pid,used_memory --format=csv
p1.wait(); p2.wait()

预期（数值待实跑回填）：

- 两个进程**各自**的 `it/s` 大约是 solo 的一半：GPU 在两个 context 之间按时间片轮转，总吞吐不变，每个人分到一半；
- `memory.used` 比 solo 时多出一整份 context（几百 MiB）加一份 a/b/c 张量：**每个进程都付了 context 的入场费**；
- `--query-compute-apps` 在 Colab 容器里可能为空或 PID 与容器内不一致（[Lab 06](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-06-nvml/) 讲过的 PID 命名空间问题）。

这就是默认模式：**公平但不并发，隔离但无配额**。如果 worker B 把 a/b 改成 12288×12288 吃掉 10 GB，A 就会 OOM，没有任何机制阻止它。

## Step 2：看见 time-slicing 的代价

把两个 worker 的矩阵改小（`512×512`），kernel 只有几十微秒，此时 context 切换的相对开销变大：

In [ ]:
!sed 's/4096/512/g' worker.py > worker_small.py
!python worker_small.py solo 10
p1 = subprocess.Popen(["python", "worker_small.py", "A", "10"])
p2 = subprocess.Popen(["python", "worker_small.py", "B", "10"])
p1.wait(); p2.wait()

预期：A + B 的合计 `it/s` **低于** solo，差值就是 time-slicing 的切换开销。kernel 越短、进程越多，这个损耗越明显。这正是推理服务（大量短 kernel）最怕的场景，也是 MPS 的动机。

## Step 3：尝试启动 MPS

MPS 的两个可执行文件 `nvidia-cuda-mps-control` 和 `nvidia-cuda-mps-server` **随驱动安装**，不属于 CUDA Toolkit：

In [ ]:
!which nvidia-cuda-mps-control nvidia-cuda-mps-server || echo "MPS binaries not in this image"

如果存在，以当前用户启动 daemon 并把两个 worker 接进去：

In [ ]:
import os
os.environ["CUDA_MPS_PIPE_DIRECTORY"] = "/tmp/nvidia-mps"
os.environ["CUDA_MPS_LOG_DIRECTORY"]  = "/tmp/nvidia-log"
!mkdir -p /tmp/nvidia-mps /tmp/nvidia-log && nvidia-cuda-mps-control -d && sleep 2 && echo "MPS control daemon started"
!echo get_server_list | nvidia-cuda-mps-control

In [ ]:
p1 = subprocess.Popen(["python", "worker_small.py", "A-mps", "10"], env=os.environ)
p2 = subprocess.Popen(["python", "worker_small.py", "B-mps", "10"], env=os.environ)
time.sleep(5)
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv
p1.wait(); p2.wait()
!echo quit | nvidia-cuda-mps-control

如果 MPS 成功接管，你会看到：

- A-mps + B-mps 的合计吞吐**回到接近 solo**，因为两个进程的 kernel 真正并发在 SM 上，不再轮流切换；
- `nvidia-smi` 的进程列表里出现 `nvidia-cuda-mps-server`，而不是两个 python 进程：**所有客户端的工作都经由这个 server 提交**，这就是“共享一个 context”的具体含义。

> **Colab 里 MPS 很可能起不来，这也是数据点**
>
> Colab 的 runtime 是一个容器，GPU 通过宿主机的驱动挂进来。MPS 需要在**拥有 GPU 的那一层**（宿主或特权容器）运行 control daemon，并通过 `/tmp/nvidia-mps` 下的管道与客户端通信。在非特权容器里，典型的失败包括：镜像里根本没有 MPS 二进制、daemon 启动后 `get_server_list` 为空、或客户端静默回落到默认模式（吞吐和 Step 2 一样）。
>
> **这正是 Kubernetes 上用 MPS 的真实困难**：MPS daemon 必须作为节点级 DaemonSet 以特权运行，所有 Pod 要挂同一个管道目录，客户端崩溃可能拖垮 server。NVIDIA device plugin 从 v0.15 起提供了 MPS 模式来封装这些细节，但它和 HAMi 一样是“平台层在替你做脏活”。如果在 Colab 里失败了，在 Metadata Card 里记录症状，然后带着这个理解去读 [L1 的 HAMi 共享作业](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/single-gpu/hami-first-share/)。

## HAMi 为什么没有直接用 MPS

MPS 解决了“并发”，但在多租户平台视角下还缺三样东西：

1. **调度器不知情**：`CUDA_MPS_ACTIVE_THREAD_PERCENTAGE` 是进程启动时的环境变量，Kubernetes 调度器无法据此决定“这个 Pod 该去哪个节点、还剩多少额度”。HAMi 的 `gpucores` / `gpumem` 是调度器可见的资源量。
2. **显存配额粗糙**：`CUDA_MPS_PINNED_DEVICE_MEM_LIMIT` 只限制 pinned 显存，且同样是静态环境变量；HAMi-core 拦截 `cudaMalloc`，在超出 `gpumem` 时直接返回 OOM，和 [Lab 04](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-04-where-did-vram-go/) 里 `nvidia-smi` 的口径一致。
3. **运维耦合**：节点常驻特权 daemon、共享管道目录、server 单点故障，都与容器“各自独立、随时重建”的生命周期相冲突。

所以 HAMi 的默认路线是 **device-plugin + HAMi-core 用户态拦截 + 调度器感知**：不改变驱动的 time-slicing，而是在它之上加一层配额。你在 [L1 HAMi 第一份共享作业](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/single-gpu/hami-first-share/) 里会亲手看到这条路线。MPS 是它的“概念原型”之一，不是实现。

## Verification

完成后你应该能：

- 说出默认 time-slicing、MPS、HAMi-core 三者在并发 / 显存 / 隔离 / 配额四个维度上的差别；
- 用 Step 1 的数据解释“每个进程都付 context 入场费”和“公平但不并发”；
- 用 Step 2 的数据解释为什么短 kernel 场景下 time-slicing 有损耗；
- 说明 MPS 解决了什么、没解决什么，以及在容器里为什么难部署；
- 说明 HAMi 为什么选择 CUDA API 拦截而不是直接用 MPS。

## Failure Modes

| 症状 | 原因 | 处理 |
| :--- | :--- | :--- |
| `nvidia-cuda-mps-control: command not found` | 容器镜像未包含驱动的 MPS 二进制 | 记录“不可用”，阅读 Step 3 的说明，继续 Step 1–2 的结论 |
| daemon 起来了但 `get_server_list` 为空 / 吞吐无变化 | 非特权容器，客户端回落到默认模式 | 同上；可在 L1 真机上重做 |
| 两个 worker 都报 `CUDA out of memory` | 4096² 的 a/b/c 加两份 context 超过 15 GiB | 不应发生（约 1.2 GiB）；若发生，先 Restart session 清掉其它 Lab 的残留 |
| `--query-compute-apps` 为空 | 容器 PID 命名空间 | 正常，用 `memory.used` 的增量推断 |
| Colab session 超时 | 免费版时长限制 | 拆小实验，见 [篇首页](../#colab-的能用多久) |

## 记录你的 Metadata Card

运行完成后，把实际数值回填到教程该章的 Metadata Card：运行日期、驱动 / CUDA 版本、关键输出数值、遇到的 Failure Mode。欢迎在 [rootsongjc/gpu-lab-notebooks](https://github.com/rootsongjc/gpu-lab-notebooks) 提交 PR 或 Issue。